## Description
```
Problem     : Regression
Algorithm   : Neural Network
Formula     : Weightes-Sum
Activation  : Relu
Loss        : MSE
Train       : Supervised
Input       : Feature and label
Output      : Probability
Dataset     : Structured : housesInfo
Tag.        : Encoding
```

## Import

In [173]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from keras.utils import to_categorical
from keras import models, layers
from keras.models import load_model
from sklearn.preprocessing  import StandardScaler
from sklearn.preprocessing import LabelBinarizer
from sklearn.preprocessing import OneHotEncoder

## Variable

In [174]:
EPOCHS = 30
BATCH_SIZE = 32
output_label = ['fire', 'nofire']

## Import dataset

In [175]:
dataset = pd.read_csv('/Volumes/data/documents/ai_document/file/dataset/housesInfo.txt', 
                      sep=" ", 
                      header=None, 
                      names=["bedrooms", "bathrooms", "area", "zipcode", "price"])

In [176]:
print("Dataset Shape: {}".format(dataset.shape))
print(dataset.head(5))

Dataset Shape: (535, 5)
   bedrooms  bathrooms  area  zipcode   price
0         4        4.0  4053    85255  869500
1         4        3.0  3343    36372  865200
2         3        4.0  3923    85266  889000
3         5        5.0  4022    85262  910000
4         3        4.0  4116    85266  971226


## Extract zipcode and count and Remove count < 25

In [177]:
zipcodes = dataset["zipcode"].value_counts().keys().tolist()
counts = dataset["zipcode"].value_counts().tolist()
print("zipcodes: {}".format(zipcodes))
print("counts: {}".format(counts))

zipcodes: [92276, 93510, 93446, 92880, 94501, 91901, 92677, 94531, 96019, 85255, 92021, 85266, 93111, 81524, 95220, 92802, 85262, 62234, 62214, 98021, 85377, 91752, 60002, 81418, 62025, 92253, 60016, 92692, 90265, 62034, 62088, 91915, 94565, 95008, 90803, 90038, 93314, 93720, 93924, 92040, 90211, 94568, 92543, 62249, 85331, 93105, 60046, 36372, 81521]
counts: [100, 60, 54, 49, 41, 32, 26, 22, 12, 12, 11, 11, 11, 11, 10, 9, 9, 7, 4, 4, 3, 3, 3, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [178]:
for (zipcode, count) in zip(zipcodes, counts):
  if count < 25:
    idxs = dataset[dataset["zipcode"] == zipcode].index
    dataset.drop(idxs, inplace=True)

In [179]:
zipcodes = dataset["zipcode"].value_counts().keys().tolist()
counts = dataset["zipcode"].value_counts().tolist()
print("zipcodes: {}".format(zipcodes))
print("counts: {}".format(counts))

zipcodes: [92276, 93510, 93446, 92880, 94501, 91901, 92677]
counts: [100, 60, 54, 49, 41, 32, 26]


In [180]:
print("Dataset Shape: {}".format(dataset.shape))
print(dataset.head(5))

Dataset Shape: (362, 5)
    bedrooms  bathrooms  area  zipcode   price
30         5        3.0  2520    93446  789000
32         3        2.0  1802    93446  365000
39         3        3.0  2146    93446  455000
80         4        2.5  2464    91901  599000
81         2        2.0  1845    91901  529800


## Separate data and label

In [190]:
x = dataset.iloc[:, :4]
y = dataset.iloc[:, 4]

In [182]:
print('X\n-------------------------------')
print(x.head(5))
print('\nY\n-------------------------------')
print(y.head(5))

X
-------------------------------
    bedrooms  bathrooms  area  zipcode
30         5        3.0  2520    93446
32         3        2.0  1802    93446
39         3        3.0  2146    93446
80         4        2.5  2464    91901
81         2        2.0  1845    91901

Y
-------------------------------
30    789000
32    365000
39    455000
80    599000
81    529800
Name: price, dtype: int64


## Separate tarin and test

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)
#y_train = np.array(y_train)
#y_test = np.array(y_test)

In [193]:
print("x_train: {}".format(x_train.shape))
print("x_test: {}".format(x_test.shape))
print("y_train: {}".format(y_train.shape))
print("y_test: {}".format(y_test.shape))

x_train: (289, 4)
x_test: (73, 4)
y_train: (289,)
y_test: (73,)


## Normalizing continuous features

In [194]:
continuous = ["bedrooms", "bathrooms", "area"]
sc = StandardScaler()
x_train_continuous = sc.fit_transform(x_train[continuous])
x_test_continuous = sc.fit_transform(x_test[continuous])

## Encoding categorical features : W1

In [195]:
encdoer = OneHotEncoder(sparse_output=False)
x_train_categorical = encdoer.fit_transform(np.array(x_train["zipcode"]).reshape(-1, 1))
x_test_categorical = encdoer.fit_transform(np.array(x_test["zipcode"]).reshape(-1, 1))

In [196]:
print(x_train_categorical)
print(x_test_categorical)

[[0. 0. 0. ... 1. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 1. 0.]
 ...
 [0. 0. 0. ... 1. 0. 0.]
 [0. 0. 0. ... 0. 1. 0.]
 [0. 0. 0. ... 0. 0. 1.]]
[[0. 0. 0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 1. 0. 0. 0.]
 [0.

## Encoding categorical features : W2

In [197]:
encdoer = LabelBinarizer()
x_train_categorical = encdoer.fit_transform(x_train["zipcode"])
x_test_categorical = encdoer.fit_transform(x_test["zipcode"])

In [198]:
print(x_train_categorical)
print(x_test_categorical)

[[0 0 0 ... 1 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 0 ... 0 1 0]
 ...
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 0 1 0]
 [0 0 0 ... 0 0 1]]
[[0 0 0 0 0 0 1]
 [1 0 0 0 0 0 0]
 [0 1 0 0 0 0 0]
 [0 0 0 0 0 1 0]
 [0 1 0 0 0 0 0]
 [0 0 0 0 0 1 0]
 [0 0 0 0 0 0 1]
 [0 1 0 0 0 0 0]
 [0 0 0 0 0 0 1]
 [0 0 0 0 0 0 1]
 [0 0 0 1 0 0 0]
 [0 0 1 0 0 0 0]
 [0 1 0 0 0 0 0]
 [0 1 0 0 0 0 0]
 [0 0 0 1 0 0 0]
 [0 0 0 1 0 0 0]
 [0 1 0 0 0 0 0]
 [0 0 0 0 0 1 0]
 [0 0 0 1 0 0 0]
 [0 0 0 1 0 0 0]
 [0 0 0 0 0 0 1]
 [0 0 0 0 0 1 0]
 [0 1 0 0 0 0 0]
 [0 0 0 0 1 0 0]
 [0 0 0 0 0 1 0]
 [0 0 0 1 0 0 0]
 [0 0 0 0 0 0 1]
 [0 0 0 1 0 0 0]
 [0 0 0 0 0 1 0]
 [0 0 0 0 0 1 0]
 [0 0 1 0 0 0 0]
 [1 0 0 0 0 0 0]
 [0 0 1 0 0 0 0]
 [0 0 0 0 0 0 1]
 [0 0 0 1 0 0 0]
 [0 1 0 0 0 0 0]
 [0 0 0 1 0 0 0]
 [0 1 0 0 0 0 0]
 [0 1 0 0 0 0 0]
 [0 0 0 0 0 1 0]
 [0 0 0 0 0 1 0]
 [0 0 0 0 0 1 0]
 [0 0 0 0 0 0 1]
 [0 1 0 0 0 0 0]
 [0 1 0 0 0 0 0]
 [0 0 0 0 0 1 0]
 [0 1 0 0 0 0 0]
 [0 0 0 0 0 1 0]
 [1 0 0 0 0 0 0]
 [0 0 0 0 1 0 0]
 [0 1 0 0 0 0 0]
 [0 0 0 0 0 1

## Concatenating continuous with categorical with features



In [199]:
x_train = np.hstack([x_train_continuous, x_train_categorical])
x_test = np.hstack([x_test_continuous, x_test_categorical])

## Normalizing label

In [200]:
maxPrice = y_train.max()
y_train_n = y_train / maxPrice
y_test_n = y_test / maxPrice

In [201]:
print("maxPrice:", maxPrice, '\n')
print(pd.DataFrame({"y_train": y_train, "y_train_n": y_train_n}))

maxPrice: 5858000 

     y_train  y_train_n
0     579500   0.098925
1     104950   0.017916
2     585000   0.099863
3     899000   0.153465
4    5858000   1.000000
..       ...        ...
284  1650000   0.281666
285   669472   0.114283
286   629000   0.107375
287   570000   0.097303
288   629000   0.107375

[289 rows x 2 columns]


In [203]:
y_train = y_train_n
y_test = y_test_n

## Model

In [204]:
mdl = models.Sequential()
mdl.add(layers.Dense(20, activation="relu"))  # Input layer
mdl.add(layers.Dense(8, activation="relu"))   # Hidden layer
mdl.add(layers.Dense(1, activation="linear")) # Output layer
mdl.compile(optimizer="SGD", loss="mse")
mdl.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_24 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Train

In [205]:
h = mdl.fit(x_train, y_train ,batch_size=BATCH_SIZE, epochs=EPOCHS, validation_data=(x_test, y_test))

Epoch 1/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1188 - val_loss: 0.0813
Epoch 2/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0826 - val_loss: 0.0525
Epoch 3/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0546 - val_loss: 0.0379
Epoch 4/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0365 - val_loss: 0.0293
Epoch 5/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0344 - val_loss: 0.0251
Epoch 6/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0263 - val_loss: 0.0215
Epoch 7/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0236 - val_loss: 0.0199
Epoch 8/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0176 - val_loss: 0.0229
Epoch 9/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0234 - val_loss: 0.0179
Epoch 10/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0229 - val_loss: 0.0162
Epoch 11/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0151 - val_loss: 0.0146
Epoch 12/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0160 - val_lo

## Loss

In [206]:
loss = mdl.evaluate(x_test, y_test)
print("Loss: {:.2f}".format(loss, loss))

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0070
Loss: 0.01


## Evaluation

In [211]:
preds = mdl.predict(x_test)
diff = preds- y_test
percentDiff = (diff / y_test) * 100
absPercentDiff = np.abs(percentDiff)
mean = np.mean(absPercentDiff)
std = np.std(absPercentDiff)
print("[INFO] mean: {:.2f}%, std: {:.2f}%".format(mean, std))

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
[INFO] mean: 200.15%, std: 308.98%
